# Prova 01 - Teoria de Markowitz

Notebook adaptado para a carteira da Ana Paula Schultze.

Ativos da Atividade 01:
- PETR4
- EMBJ3
- WEGE3
- ABEV3
- BPAC11

Pesos originais da Atividade 01:
- PETR4: 35%
- EMBJ3: 20%
- WEGE3: 15%
- ABEV3: 20%
- BPAC11: 10%


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize
from matplotlib.ticker import PercentFormatter
from IPython.display import display

plt.style.use('seaborn-v0_8')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


In [ ]:
tickers = ['PETR4.SA', 'EMBJ3.SA', 'WEGE3.SA', 'ABEV3.SA', 'BPAC11.SA']
nomes_tickers = {
    'PETR4.SA': 'PETR4',
    'EMBJ3.SA': 'EMBJ3',
    'WEGE3.SA': 'WEGE3',
    'ABEV3.SA': 'ABEV3',
    'BPAC11.SA': 'BPAC11',
}

pesos_atividade_01 = pd.Series({
    'PETR4.SA': 0.35,
    'EMBJ3.SA': 0.20,
    'WEGE3.SA': 0.15,
    'ABEV3.SA': 0.20,
    'BPAC11.SA': 0.10,
})

retorno_atividade_01 = -0.004003866891771016
ativo_livre_risco = 0.0290

data_inicio_atividade = pd.Timestamp('2026-03-03')
data_fim_atividade = pd.Timestamp('2026-03-23')

data_fim_36m = pd.Timestamp('2026-06-29')
data_inicio_36m = data_fim_36m - pd.DateOffset(months=36)

print('Ativos analisados:', [nomes_tickers[t] for t in tickers])
print('Periodo Atividade 01:', data_inicio_atividade.date(), 'ate', data_fim_atividade.date())
print('Periodo 36 meses:', data_inicio_36m.date(), 'ate', data_fim_36m.date())


In [ ]:
def baixar_precos(tickers, start, end):
    series = []

    for ticker in tickers:
        dados = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False, threads=False)

        if dados.empty:
            raise ValueError(f'Nao foi possivel baixar dados para {ticker}.')

        if isinstance(dados.columns, pd.MultiIndex):
            coluna = 'Adj Close' if 'Adj Close' in dados.columns.get_level_values(0) else 'Close'
            serie = dados[coluna].iloc[:, 0].rename(ticker)
        else:
            coluna = 'Adj Close' if 'Adj Close' in dados.columns else 'Close'
            serie = dados[coluna].rename(ticker)
        series.append(serie)

    precos = pd.concat(series, axis=1).sort_index()
    return precos.dropna(how='any')


def calcular_retorno_e_covariancia(precos):
    retornos_diarios = precos.pct_change(fill_method=None).dropna()
    retornos_anualizados = retornos_diarios.mean() * 252
    covariancia_anualizada = retornos_diarios.cov() * 252
    return retornos_diarios, retornos_anualizados, covariancia_anualizada


def calcula_retorno(carteira, retornos):
    return float(np.sum(np.asarray(carteira) * np.asarray(retornos)))


def calcula_risco(carteira, covariancia):
    carteira = np.asarray(carteira)
    covariancia = np.asarray(covariancia)
    return float(np.sqrt(np.dot(carteira.T, np.dot(covariancia, carteira))))


def calcula_fronteira_eficiente(retornos, covariancia, num_pontos=5000, seed=42):
    rng = np.random.default_rng(seed)
    num_ativos = len(retornos)
    pesos = rng.random((num_pontos, num_ativos))
    pesos /= pesos.sum(axis=1, keepdims=True)

    riscos = np.array([calcula_risco(w, covariancia) for w in pesos])
    retornos_carteira = np.array([calcula_retorno(w, retornos) for w in pesos])
    sharpe = np.divide(
        retornos_carteira - ativo_livre_risco,
        riscos,
        out=np.full_like(retornos_carteira, np.nan),
        where=riscos > 0
    )

    return riscos, retornos_carteira, sharpe


def calcula_minima_variancia(retornos, covariancia, epsilon=1e-8):
    num_ativos = len(retornos)
    ones = np.ones(num_ativos)
    cov_ajustada = np.asarray(covariancia) + np.eye(num_ativos) * epsilon
    cov_inv = np.linalg.inv(cov_ajustada)
    pesos = np.dot(cov_inv, ones) / np.dot(ones, np.dot(cov_inv, ones))
    return pesos


def calcula_carteira_otima(retornos, covariancia, ativo_livre_risco, epsilon=1e-8):
    num_ativos = len(retornos)
    cov_ajustada = np.asarray(covariancia) + np.eye(num_ativos) * epsilon
    w0 = np.ones(num_ativos) / num_ativos
    bounds = tuple((0, 1) for _ in range(num_ativos))

    def neg_sharpe_ratio(weights):
        retorno = calcula_retorno(weights, retornos)
        risco = calcula_risco(weights, cov_ajustada)
        return -(retorno - ativo_livre_risco) / risco

    resultado = minimize(
        neg_sharpe_ratio,
        w0,
        method='SLSQP',
        bounds=bounds,
        constraints=({'type': 'eq', 'fun': lambda x: np.sum(x) - 1},)
    )

    return resultado.x


def retornos_simples_periodo(precos):
    return (precos.iloc[-1] / precos.iloc[0]) - 1


def resumo_carteira(titulo, tickers, pesos, retornos_anualizados, covariancia, retornos_periodo=None):
    tabela = pd.DataFrame({
        'Ticker': [nomes_tickers[t] for t in tickers],
        'Peso': pesos,
        'Retorno anualizado do ativo': [retornos_anualizados[t] for t in tickers],
    })

    if retornos_periodo is not None:
        tabela['Retorno simples no periodo'] = [retornos_periodo[t] for t in tickers]

    retorno_anualizado = calcula_retorno(pesos, retornos_anualizados[tickers].values)
    risco_anualizado = calcula_risco(pesos, covariancia.loc[tickers, tickers].values)

    print(f'\n{titulo}')
    display(tabela)
    print(f'Retorno anualizado da carteira: {retorno_anualizado:.4%}')
    print(f'Risco anualizado da carteira: {risco_anualizado:.4%}')

    if retornos_periodo is not None:
        retorno_periodo = calcula_retorno(pesos, retornos_periodo[tickers].values)
        print(f'Retorno simples no periodo: {retorno_periodo:.4%}')

    return tabela


def calcula_max_drawdown(retornos):
    riqueza = (1 + retornos).cumprod()
    pico = riqueza.cummax()
    drawdown = riqueza / pico - 1
    return float(drawdown.min())


def simular_rebalanceamento_min_variancia(precos, meses_rebalanceamento, janela_estimacao_meses=12):
    retornos_diarios = precos.pct_change(fill_method=None).dropna()
    grupos_mensais = list(retornos_diarios.groupby(retornos_diarios.index.to_period('M')))

    retornos_carteira = []
    historico_pesos = []
    turnover = []
    pesos_anteriores = None

    for i in range(janela_estimacao_meses - 1, len(grupos_mensais) - 1, meses_rebalanceamento):
        janela = grupos_mensais[i - janela_estimacao_meses + 1:i + 1]
        dados_janela = pd.concat([grupo for _, grupo in janela])

        retornos_janela = dados_janela.mean() * 252
        cov_janela = dados_janela.cov() * 252
        pesos = calcula_minima_variancia(retornos_janela.values, cov_janela.values)

        proximo_indice = min(i + meses_rebalanceamento, len(grupos_mensais) - 1)
        periodo_pos_rebalanceamento = grupos_mensais[i + 1:proximo_indice + 1]

        if not periodo_pos_rebalanceamento:
            continue

        dados_hold = pd.concat([grupo for _, grupo in periodo_pos_rebalanceamento])
        serie_carteira = dados_hold.dot(pesos)
        retornos_carteira.append(serie_carteira)

        data_rebalanceamento = periodo_pos_rebalanceamento[0][1].index[0]
        historico_pesos.append(pd.Series(pesos, index=precos.columns, name=data_rebalanceamento))

        if pesos_anteriores is not None:
            turnover.append(np.abs(pesos - pesos_anteriores).sum())

        pesos_anteriores = pesos

    serie_resultado = pd.concat(retornos_carteira).sort_index()
    riqueza = (1 + serie_resultado).cumprod()

    retorno_anualizado = (1 + serie_resultado).prod() ** (252 / len(serie_resultado)) - 1
    volatilidade_anualizada = serie_resultado.std() * np.sqrt(252)
    sharpe = np.nan

    if volatilidade_anualizada > 0:
        sharpe = (retorno_anualizado - ativo_livre_risco) / volatilidade_anualizada

    resultado = {
        'retornos_diarios': serie_resultado,
        'riqueza': riqueza,
        'pesos': pd.DataFrame(historico_pesos),
        'retorno_anualizado': retorno_anualizado,
        'volatilidade_anualizada': volatilidade_anualizada,
        'sharpe': sharpe,
        'turnover_medio': float(np.mean(turnover)) if turnover else 0.0,
        'max_drawdown': calcula_max_drawdown(serie_resultado),
    }

    return resultado


## Questao 1

Mesmos ativos e mesmo periodo da Atividade 01.

Observacao: para a comparacao do item (b), o notebook calcula tambem o retorno simples do periodo, porque o retorno da planilha da Atividade 01 foi calculado com base no preco inicial e final do periodo, e nao como retorno anualizado.


In [ ]:
precos_q1 = baixar_precos(
    tickers,
    start=data_inicio_atividade.strftime('%Y-%m-%d'),
    end=(data_fim_atividade + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
)

retornos_diarios_q1, retornos_anualizados_q1, covariancia_q1 = calcular_retorno_e_covariancia(precos_q1)
retornos_periodo_q1 = retornos_simples_periodo(precos_q1)

plt.figure(figsize=(12, 6))
for ticker in tickers:
    plt.plot(precos_q1.index, precos_q1[ticker], label=nomes_tickers[ticker])
plt.title('Questao 1 - Evolucao dos precos no periodo da Atividade 01')
plt.xlabel('Data')
plt.ylabel('Preco ajustado')
plt.legend()
plt.grid(True)
plt.show()

resumo_ativos_q1 = pd.DataFrame({
    'Ticker': [nomes_tickers[t] for t in tickers],
    'Retorno anualizado': [retornos_anualizados_q1[t] for t in tickers],
    'Risco anualizado': [retornos_diarios_q1[t].std() * np.sqrt(252) for t in tickers],
    'Retorno simples no periodo': [retornos_periodo_q1[t] for t in tickers],
})

display(resumo_ativos_q1)

riscos_fronteira_q1, retornos_fronteira_q1, sharpe_fronteira_q1 = calcula_fronteira_eficiente(
    retornos_anualizados_q1.values, covariancia_q1.values
)

pesos_min_var_q1 = calcula_minima_variancia(retornos_anualizados_q1.values, covariancia_q1.values)
pesos_otimos_q1 = calcula_carteira_otima(retornos_anualizados_q1.values, covariancia_q1.values, ativo_livre_risco)

retorno_min_var_q1 = calcula_retorno(pesos_min_var_q1, retornos_anualizados_q1.values)
risco_min_var_q1 = calcula_risco(pesos_min_var_q1, covariancia_q1.values)
retorno_otimo_q1 = calcula_retorno(pesos_otimos_q1, retornos_anualizados_q1.values)
risco_otimo_q1 = calcula_risco(pesos_otimos_q1, covariancia_q1.values)

plt.figure(figsize=(10, 6))
plt.scatter(riscos_fronteira_q1, retornos_fronteira_q1, c=sharpe_fronteira_q1, s=18, cmap='viridis')
plt.scatter(risco_min_var_q1, retorno_min_var_q1, color='red', marker='*', s=220, label='Minima variancia')
plt.scatter(risco_otimo_q1, retorno_otimo_q1, color='green', marker='*', s=220, label='Carteira otima')
for ticker in tickers:
    risco_ativo = retornos_diarios_q1[ticker].std() * np.sqrt(252)
    retorno_ativo = retornos_anualizados_q1[ticker]
    plt.scatter(risco_ativo, retorno_ativo, color='black', s=40)
    plt.annotate(nomes_tickers[ticker], (risco_ativo, retorno_ativo), xytext=(5, 5), textcoords='offset points')
plt.title('Questao 1 - Fronteira eficiente')
plt.xlabel('Risco anualizado')
plt.ylabel('Retorno anualizado')
plt.colorbar(label='Indice de Sharpe')
plt.legend()
plt.grid(True)
plt.show()

resumo_carteira('Carteira original da Atividade 01', tickers, pesos_atividade_01.values, retornos_anualizados_q1, covariancia_q1, retornos_periodo_q1)
resumo_carteira('Carteira de minima variancia - Questao 1', tickers, pesos_min_var_q1, retornos_anualizados_q1, covariancia_q1, retornos_periodo_q1)
resumo_carteira('Carteira otima - referencia adicional', tickers, pesos_otimos_q1, retornos_anualizados_q1, covariancia_q1, retornos_periodo_q1)

retorno_periodo_min_var_q1 = calcula_retorno(pesos_min_var_q1, retornos_periodo_q1.values)
diferenca_retorno_q1 = retorno_periodo_min_var_q1 - retorno_atividade_01

print('\nComparacao pedida na Questao 1(b)')
print(f'Retorno da Atividade 01: {retorno_atividade_01:.4%}')
print(f'Retorno da carteira de minima variancia no mesmo periodo: {retorno_periodo_min_var_q1:.4%}')
print(f'Diferenca entre os retornos: {diferenca_retorno_q1:.4%}')

if diferenca_retorno_q1 > 0:
    print('Comentario: a carteira de minima variancia entregou retorno superior ao da carteira original da Atividade 01 nesse intervalo.')
else:
    print('Comentario: a carteira de minima variancia entregou retorno inferior ao da carteira original da Atividade 01 nesse intervalo.')

print('Em compensacao, a carteira de minima variancia foi montada para reduzir risco, entao a avaliacao deve considerar o par retorno-risco e nao apenas o retorno isolado.')


## Questao 2

Analise dos ultimos 36 meses para os mesmos ativos da Atividade 01.


In [ ]:
precos_36m = baixar_precos(
    tickers,
    start=data_inicio_36m.strftime('%Y-%m-%d'),
    end=(data_fim_36m + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
)

retornos_diarios_36m, retornos_anualizados_36m, covariancia_36m = calcular_retorno_e_covariancia(precos_36m)
pesos_min_var_36m = calcula_minima_variancia(retornos_anualizados_36m.values, covariancia_36m.values)

print('Carteira de minima variancia do periodo completo de 36 meses')
resumo_carteira('Questao 2(a) - Periodo completo', tickers, pesos_min_var_36m, retornos_anualizados_36m, covariancia_36m)

pesos_mensais = {}
for periodo, dados_mes in retornos_diarios_36m.groupby(retornos_diarios_36m.index.to_period('M')):
    if len(dados_mes) < 5:
        continue

    retornos_mes = dados_mes.mean() * 252
    cov_mes = dados_mes.cov() * 252
    pesos_mes = calcula_minima_variancia(retornos_mes.values, cov_mes.values)
    pesos_mensais[str(periodo)] = pesos_mes

pesos_mensais_df = pd.DataFrame(pesos_mensais, index=tickers).T
pesos_mensais_df.columns = [nomes_tickers[t] for t in tickers]

print('\nQuestao 2(a) - Carteira de minima variancia mes a mes')
display(pesos_mensais_df)

estatisticas_pesos = pd.DataFrame({
    'Media dos pesos mensais': pesos_mensais_df.mean(),
    'Desvio padrao dos pesos mensais': pesos_mensais_df.std(),
})

print('\nQuestao 2(b) - Media e desvio padrao dos pesos mensais')
display(estatisticas_pesos)

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 12))
axes = axes.flatten()

for i, ticker in enumerate(pesos_mensais_df.columns):
    axes[i].hist(pesos_mensais_df[ticker], bins=10, edgecolor='black', color='steelblue')
    axes[i].set_title(f'Questao 2(c) - Histograma dos pesos de {ticker}')
    axes[i].set_xlabel('Peso W')
    axes[i].set_ylabel('Frequencia')
    axes[i].xaxis.set_major_formatter(PercentFormatter(1.0))

axes[-1].axis('off')
plt.tight_layout()
plt.show()


## Questao 3

Metodologia proposta para definir o intervalo ideal de rebalanceamento:

1. Usar os ultimos 36 meses de dados diarios reais.
2. Estimar a carteira de minima variancia com uma janela movel de 12 meses.
3. Simular diferentes frequencias de rebalanceamento: mensal, bimestral, trimestral, semestral e anual.
4. Comparar o desempenho realizado fora da amostra por quatro metricas: retorno anualizado, volatilidade anualizada, indice de Sharpe e turnover medio.
5. Escolher como intervalo ideal aquele com menor volatilidade realizada, porque o objetivo da estrategia e minimizar risco. Em caso de resultados muito proximos, o desempate privilegia menor turnover, por representar menor necessidade de trocas e menor custo operacional.


In [ ]:
frequencias = {
    'Mensal': 1,
    'Bimestral': 2,
    'Trimestral': 3,
    'Semestral': 6,
    'Anual': 12,
}

simulacoes = {}
for nome, meses in frequencias.items():
    simulacoes[nome] = simular_rebalanceamento_min_variancia(precos_36m, meses_rebalanceamento=meses, janela_estimacao_meses=12)

resumo_rebalanceamento = pd.DataFrame({
    nome: {
        'Retorno anualizado': resultado['retorno_anualizado'],
        'Volatilidade anualizada': resultado['volatilidade_anualizada'],
        'Indice de Sharpe': resultado['sharpe'],
        'Turnover medio': resultado['turnover_medio'],
        'Max drawdown': resultado['max_drawdown'],
    }
    for nome, resultado in simulacoes.items()
}).T

resumo_rebalanceamento = resumo_rebalanceamento.sort_values(
    by=['Volatilidade anualizada', 'Turnover medio', 'Indice de Sharpe'],
    ascending=[True, True, False]
)

display(resumo_rebalanceamento)

melhor_intervalo = resumo_rebalanceamento.index[0]
print(f'Intervalo sugerido para rebalanceamento: {melhor_intervalo}')
print('Justificativa automatica: esse intervalo apresentou a menor volatilidade realizada e, no criterio de desempate, menor turnover e melhor Sharpe relativo.')

plt.figure(figsize=(12, 6))
for nome, resultado in simulacoes.items():
    plt.plot(resultado['riqueza'].index, resultado['riqueza'], label=nome)
plt.title('Questao 3 - Evolucao do capital por frequencia de rebalanceamento')
plt.xlabel('Data')
plt.ylabel('Capital acumulado')
plt.legend()
plt.grid(True)
plt.show()

print('\nConclusao sugerida para o relatorio:')
print(f'- Como a estrategia analisada e de minima variancia, o criterio principal foi a menor volatilidade realizada fora da amostra.')
print(f'- Entre as frequencias testadas, {melhor_intervalo} foi o melhor compromisso entre estabilidade da carteira e necessidade de rebalancear.')
print('- Portanto, esse intervalo pode ser defendido como o intervalo de tempo ideal dentro da metodologia proposta, com base em dados reais dos ultimos 36 meses.')
